# IoT-NetGuard — Model 1: Naïve Bayes
**Project:** Anomaly Detection for IoT Cybersecurity  
**Dataset:** `iot23_combined.csv` (output of `01_Data_Preprocessing.ipynb`)  
**Algorithm:** Gaussian Naïve Bayes — probabilistic baseline classifier

## 1. Imports

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
print('All imports successful.')

## 2. Load Dataset

In [ ]:
FILEPATH = "iot23_combined.csv"
FEATURES = [
    'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
    'proto_icmp', 'proto_tcp', 'proto_udp',
    'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0',
    'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1',
    'conn_state_S2', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR'
]
TARGET = 'label'

df = pd.read_csv(FILEPATH)
# Remove index column if present
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print(f'Dataset shape : {df.shape}')
print(f'Label distribution:\n{df[TARGET].value_counts().to_string()}')

## 3. Feature Selection & Normalisation

In [ ]:
X = df[FEATURES].values
Y = df[TARGET].values

# Min-Max normalisation (required for Gaussian NB to work well)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=100, stratify=Y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing  samples : {X_test.shape[0]:,}')

## 4. Train Gaussian Naïve Bayes

In [ ]:
print('Training Gaussian Naïve Bayes...')
start = time.time()

clf = GaussianNB()
clf.fit(X_train, Y_train)

elapsed = time.time() - start
print(f'Training complete in {elapsed:.2f} seconds')

## 5. Evaluation

In [ ]:
Y_pred = clf.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)

print(f'Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Time cost : {elapsed:.2f} seconds')
print()
print('Classification Report:')
print(classification_report(Y_test, Y_pred))

## 6. Confusion Matrix

In [ ]:
labels = sorted(df[TARGET].unique())
cm = confusion_matrix(Y_test, Y_pred, labels=labels)

plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Naïve Bayes — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('outputs/nb_confusion_matrix.png', dpi=150)
plt.show()
print('Saved → outputs/nb_confusion_matrix.png')

## 7. Summary

In [ ]:
print('=' * 45)
print('  IoT-NetGuard — Naïve Bayes Summary')
print('=' * 45)
print(f'  Model     : Gaussian Naïve Bayes')
print(f'  Features  : {len(FEATURES)}')
print(f'  Train/Test: 80% / 20%')
print(f'  Accuracy  : {accuracy*100:.2f}%')
print(f'  Time      : {elapsed:.2f}s')
print('=' * 45)